## MS&E 226 — Feature Engineering & Two-Stage Subset Selection (UCI Online Shoppers)

**Goal:** Find a defensible, parsimonious feature set.  
**Method:** 
1) Engineer behaviorally meaningful features (breadth, depth, value, bounce).  
2) Check redundancy (numeric & technical categoricals).  
3) **Stage 1:** Compare alternative *numeric* representations (raw vs ratios vs page-type).  
4) **Stage 2:** Add temporal/user/technical on top of the best numeric set.  
5) Evaluate with 5-fold CV using Logistic Regression (linear) and Random Forest (non-linear).  
**Why:** Separates “feature quality” from “model complexity” and avoids the cop-out of “use everything.”


### Imports & setup (Code + short note)

In [34]:
import os, warnings, json
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_validate
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore")
np.random.seed(42)

DATA_PATH = "../data/train_data.csv"
RESULTS_DIR = "../results"; os.makedirs(RESULTS_DIR, exist_ok=True)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


### Load data & quick sanity

In [35]:
df = pd.read_csv(DATA_PATH)
assert "Revenue" in df.columns, "Expected 'Revenue' target."
y = df["Revenue"].astype(int)
X_raw = df.drop(columns=["Revenue"])

print(f"Rows: {len(df)} | X cols: {X_raw.shape[1]} | Positive rate: {y.mean():.2%}")
X_raw.head(3)


Rows: 9864 | X cols: 17 | Positive rate: 15.47%


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend
0,3,142.500000,0,0.00,48,1052.255952,0.004348,0.013043,0.000000,0.0,Nov,1,8,6,11,Returning_Visitor,False
1,6,437.391304,2,235.55,83,2503.881781,0.002198,0.004916,2.086218,0.0,Mar,2,2,3,2,Returning_Visitor,False
2,1,41.125000,0,0.00,126,4310.004668,0.000688,0.012823,3.451072,0.0,Nov,2,2,2,2,Returning_Visitor,False


### Base schema groups & dtype hygiene

In [36]:
numeric_features = [
    "Administrative","Administrative_Duration",
    "Informational","Informational_Duration",
    "ProductRelated","ProductRelated_Duration",
    "BounceRates","ExitRates","PageValues","SpecialDay"
]
categorical_features = ["Month","VisitorType"]
id_like_features = ["OperatingSystems","Browser","Region","TrafficType"]  # treat as cats
boolean_features = ["Weekend"]

# Coerce/clean
for c in numeric_features: X_raw[c] = pd.to_numeric(X_raw[c], errors="coerce")
X_raw["Weekend"] = X_raw["Weekend"].astype(int)
for c in categorical_features:
    if X_raw[c].isna().any(): X_raw[c] = X_raw[c].fillna("Unknown")
for c in id_like_features:
    if X_raw[c].isna().any(): X_raw[c] = X_raw[c].fillna(-1).astype(int)

X_raw.head(3)


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend
0,3,142.500000,0,0.00,48,1052.255952,0.004348,0.013043,0.000000,0.0,Nov,1,8,6,11,Returning_Visitor,0
1,6,437.391304,2,235.55,83,2503.881781,0.002198,0.004916,2.086218,0.0,Mar,2,2,3,2,Returning_Visitor,0
2,1,41.125000,0,0.00,126,4310.004668,0.000688,0.012823,3.451072,0.0,Nov,2,2,2,2,Returning_Visitor,0


### Engineer features (breadth, depth, value, bounce)

**Why**: Capture behavior meaningfully; we’ll prune redundancy after.

In [37]:
def engineer_features(X: pd.DataFrame) -> pd.DataFrame:
    Xf = X.copy()
    total_pages = Xf["Administrative"] + Xf["Informational"] + Xf["ProductRelated"] + 1e-5
    total_dur   = Xf["Administrative_Duration"] + Xf["Informational_Duration"] + Xf["ProductRelated_Duration"]

    # Breadth & depth
    Xf["total_pages"]       = total_pages
    Xf["avg_time_per_page"] = total_dur / total_pages

    # Page-type depth (experimental variants)
    Xf["avg_time_per_product_page"] = Xf["ProductRelated_Duration"]   / (Xf["ProductRelated"]   + 1e-5)
    Xf["avg_time_per_info_page"]    = Xf["Informational_Duration"]    / (Xf["Informational"]    + 1e-5)
    Xf["avg_time_per_admin_page"]   = Xf["Administrative_Duration"]   / (Xf["Administrative"]   + 1e-5)

    # Focus composition
    Xf["product_focus"] = Xf["ProductRelated"] / total_pages
    Xf["info_focus"]    = Xf["Informational"]  / total_pages
    Xf["admin_focus"]   = Xf["Administrative"] / total_pages

    # Bounce & value
    Xf["bounce_exit_ratio"]   = Xf["BounceRates"] / (Xf["ExitRates"] + 1e-5)
    Xf["bounce_exit_product"] = Xf["BounceRates"] *  Xf["ExitRates"]
    Xf["value_per_page"]      = Xf["PageValues"] / total_pages
    # (intentionally DROP total_duration; it’s redundant with total_pages × avg_time_per_page)

    Xf["Weekend"] = Xf["Weekend"].astype(int)
    return Xf

X_engineered = engineer_features(X_raw)

derived_all = [
    "total_pages","avg_time_per_page",
    "avg_time_per_product_page","avg_time_per_info_page","avg_time_per_admin_page",
    "product_focus","info_focus","admin_focus",
    "bounce_exit_ratio","bounce_exit_product",
    "value_per_page"
]
print(f"Derived features created: {len(derived_all)}")
X_engineered[derived_all].head(3)


Derived features created: 11


,total_pages,avg_time_per_page,avg_time_per_product_page,avg_time_per_info_page,avg_time_per_admin_page,product_focus,info_focus,admin_focus,bounce_exit_ratio,bounce_exit_product,value_per_page
0,51.00001,23.426583,21.921994,0.000000,47.499842,0.941176,0.000000,0.058824,0.333078,0.000057,0.000000
1,91.00001,34.910140,30.167247,117.774411,72.898429,0.912088,0.021978,0.065934,0.446151,0.000011,0.022925
2,127.00001,34.260861,34.206384,0.000000,41.124589,0.992126,0.000000,0.007874,0.053599,0.000009,0.027174


In [38]:
# --- Build a canonical dataset that includes ORIGINAL + ENGINEERED + TARGET ---

# 0) Ensure target is aligned
y = df["Revenue"].astype(int)  # or however you named the original df with target

# 1) Explicit column order (original blocks first, then engineered)
base_cols_order = (
    numeric_features
    + id_like_features
    + categorical_features
    + boolean_features
)

engineered_cols_order = derived_all  # keep ALL engineered (even ones we might drop later)

# 2) Keep only columns that exist; preserve order; avoid dups
def unique_in_order(cols): 
    return list(dict.fromkeys([c for c in cols if c in X_engineered.columns]))

final_cols = unique_in_order(base_cols_order) + unique_in_order(engineered_cols_order)

# 3) Assemble features + target
features_all = X_engineered[final_cols].copy()
features_all["Revenue"] = y.values  # append target as last column

print(f"All-features table shape: {features_all.shape}")
print("First 12 columns:", list(features_all.columns[:12]))
print("Last columns:", list(features_all.columns[-5:]))

# 4) Save two files:
#    a) with target (for quick EDA / splits), b) features only (for inference pipelines)
features_all.to_csv("../data/train_data_with_engineered.csv", index=False)
features_all.drop(columns=["Revenue"]).to_csv("../data/train_features_only.csv", index=False)

print("✅ Saved:")
print("  • ../data/train_data_with_engineered.csv (features + Revenue)")
print("  • ../data/train_features_only.csv (features only)")


All-features table shape: (9864, 29)
First 12 columns: ['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'OperatingSystems', 'Browser']
Last columns: ['admin_focus', 'bounce_exit_ratio', 'bounce_exit_product', 'value_per_page', 'Revenue']
✅ Saved:
  • ../data/train_data_with_engineered.csv (features + Revenue)
  • ../data/train_features_only.csv (features only)


### Numeric redundancy peek

In [39]:
num_like = list(set(numeric_features + derived_all + boolean_features))
corr_abs = X_engineered[num_like].corr().abs()
pairs = []
thr = 0.85
for i,a in enumerate(corr_abs.columns):
    for j in range(i+1, len(corr_abs.columns)):
        b = corr_abs.columns[j]
        v = corr_abs.loc[a,b]
        if pd.notnull(v) and v >= thr:
            pairs.append((a,b,float(v)))
redundancy_df = pd.DataFrame(pairs, columns=["feature_a","feature_b","|corr|"]).sort_values("|corr|", ascending=False)
redundancy_df.head(15)


,feature_a,feature_b,|corr|
7,ProductRelated,total_pages,0.997169
4,avg_time_per_page,avg_time_per_product_page,0.979600
3,BounceRates,bounce_exit_product,0.978624
2,admin_focus,product_focus,0.914468
0,ExitRates,BounceRates,0.912273
1,ExitRates,bounce_exit_product,0.889356
5,ProductRelated_Duration,ProductRelated,0.857793
6,ProductRelated_Duration,total_pages,0.857000


We shouldnt take totals and produt related things. We should either just take totals, or take the breakdowns of them. We will try both and see which perfroms better

### Technical (categorical) quick check (Code)

In [40]:
categorical_cols = ["OperatingSystems","Browser","Region","TrafficType"]

# 1) Dominance: if top category >85%, likely low value
for col in categorical_cols:
    top_share = X_engineered[col].value_counts(normalize=True).iloc[0]
    print(f"{col:18s} | top category share = {top_share:.2%}")

# 2) Rough overlap via factorized correlation (good enough for redundancy)
encoded = pd.DataFrame({c: pd.factorize(X_engineered[c])[0] for c in categorical_cols})
cat_corr = encoded.corr().round(2)
print("\nRough categorical correlation matrix:")
display(cat_corr)

redundant_cats = [(a,b,cat_corr.loc[a,b]) for i,a in enumerate(cat_corr.columns)
                  for b in cat_corr.columns[i+1:] if abs(cat_corr.loc[a,b])>0.7]
print("\nPotentially redundant categorical pairs (|r|>0.7):")
for a,b,r in redundant_cats: print(f"  {a} ↔ {b}: r={r:.2f}")


OperatingSystems   | top category share = 52.97%
Browser            | top category share = 64.49%
Region             | top category share = 38.48%
TrafficType        | top category share = 31.61%

Rough categorical correlation matrix:


,OperatingSystems,Browser,Region,TrafficType
OperatingSystems,1.00,-0.30,0.03,0.07
Browser,-0.30,1.00,0.00,-0.04
Region,0.03,0.00,1.00,-0.01
TrafficType,0.07,-0.04,-0.01,1.00



Potentially redundant categorical pairs (|r|>0.7):


This tells us that these 4 featues re largely indepednent so we cna inclue all of them if we want

### Finalize a pruned engineered set

In [41]:
pruned_derived = [
    # Breadth + depth
    "total_pages","avg_time_per_page",
    # Page-type depth signals (keep 1–2 if helpful)
    "avg_time_per_product_page","avg_time_per_info_page",
    # Value + bounce summaries
    "value_per_page","bounce_exit_ratio",
    # Focus composition (keep product + info; admin is complement)
    "product_focus","info_focus"
]
print(f"{len(pruned_derived)} engineered features retained:", pruned_derived)


8 engineered features retained: ['total_pages', 'avg_time_per_page', 'avg_time_per_product_page', 'avg_time_per_info_page', 'value_per_page', 'bounce_exit_ratio', 'product_focus', 'info_focus']


### Build two-stage experiment registry

In [42]:
# Stage 1: numeric-only candidates (pick a winner)
numeric_variants = {
    "raw_counts": ["Administrative","Informational","ProductRelated"],
    "raw_durations": ["Administrative_Duration","Informational_Duration","ProductRelated_Duration"],
    "breadth_depth": ["total_pages","avg_time_per_page"],
    "page_type_depth": ["avg_time_per_product_page","avg_time_per_info_page"],
    "behavioral_ratios": ["product_focus","info_focus","bounce_exit_ratio","value_per_page"],
    "pruned_engineered": pruned_derived,
}

# Stage 2 context blocks
context_blocks = {
    "temporal": ["Month","SpecialDay","Weekend"],
    "user": ["VisitorType"],
    "technical": ["OperatingSystems","Browser","Region","TrafficType"],
}

# Build feature_sets: stage 1 and stage 2 expansions
feature_sets = {}

# Stage 1: numeric only
for name, cols in numeric_variants.items():
    feature_sets[f"num_{name}"] = cols

# Stage 2: add context to each numeric base
for base_name, base_cols in numeric_variants.items():
    for ctx_name, ctx_cols in context_blocks.items():
        feature_sets[f"{base_name}_plus_{ctx_name}"] = base_cols + ctx_cols

# (Optional) combined temporal + user layer on pruned core
feature_sets["pruned_plus_temporal_user"] = numeric_variants["pruned_engineered"] + context_blocks["temporal"] + context_blocks["user"]

# Summarize
pd.DataFrame([{"Feature_Set": k, "n_cols": len(v)} for k,v in feature_sets.items()]).sort_values("Feature_Set").head(12)


,Feature_Set,n_cols
20,behavioral_ratios_plus_technical,8
18,behavioral_ratios_plus_temporal,7
19,behavioral_ratios_plus_user,5
14,breadth_depth_plus_technical,6
12,breadth_depth_plus_temporal,5
13,breadth_depth_plus_user,3
4,num_behavioral_ratios,4
2,num_breadth_depth,2
3,num_page_type_depth,2
5,num_pruned_engineered,8


### Sanitize (dedupe/validate) before modeling

In [43]:
from collections import Counter

def dedupe_preserve_order(cols): return list(dict.fromkeys(cols))
def validate_set(name, cols, df):
    dups = {c:cnt for c,cnt in Counter(cols).items() if cnt>1}
    if dups: print(f"[{name}] duplicates -> {dups}")
    unique = dedupe_preserve_order(cols)
    missing = [c for c in unique if c not in df.columns]
    if missing: print(f"[{name}] missing -> {missing}")
    return unique

feature_sets_clean = {n: validate_set(n, c, X_engineered) for n,c in feature_sets.items()}


### Preprocessor & models

In [44]:
def make_preprocessor(cols):
    num = [c for c in cols if c in (set(numeric_features) | set(derived_all) | set(boolean_features))]
    cat = [c for c in cols if c in (set(categorical_features) | set(id_like_features))]
    return ColumnTransformer([
        ("num", StandardScaler(), num),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat)
    ], remainder="drop")

models = {
    "LogReg": LogisticRegression(max_iter=2000, solver="lbfgs"),
    "RF": RandomForestClassifier(n_estimators=200, min_samples_leaf=2, random_state=42, n_jobs=-1)
}


### Stage 1: Evaluate numeric-only variants

In [45]:
rows1 = []
for fs_name, fs_cols in feature_sets_clean.items():
    if not fs_name.startswith("num_"): continue
    pre = make_preprocessor(fs_cols)
    for mdl_name, mdl in models.items():
        pipe = Pipeline([("prep", pre), ("clf", mdl)])
        f1  = cross_val_score(pipe, X_engineered[fs_cols], y, cv=cv, scoring="f1", n_jobs=-1)
        acc = cross_val_score(pipe, X_engineered[fs_cols], y, cv=cv, scoring="accuracy", n_jobs=-1)
        rows1.append({"Feature_Set":fs_name,"Model":mdl_name,"Mean_F1":f1.mean(),"Mean_Acc":acc.mean()})

stage1 = pd.DataFrame(rows1).sort_values(["Model","Mean_F1"], ascending=[True,False])
stage1_wide = stage1.pivot(index="Feature_Set", columns="Model", values="Mean_F1").sort_values("RF", ascending=False)
display(stage1_wide)
best_numeric_set = stage1.groupby("Feature_Set")["Mean_F1"].mean().idxmax()
print("Best numeric engine:", best_numeric_set)


Model,LogReg,RF
Feature_Set,,
num_pruned_engineered,0.464492,0.610695
num_behavioral_ratios,0.452971,0.586526
num_breadth_depth,0.030048,0.112173
num_page_type_depth,0.001299,0.083283
num_raw_counts,0.026217,0.074909
num_raw_durations,0.031592,0.050419


Best numeric engine: num_pruned_engineered


### Stage 2: Context lift on top-k numeric bases

In [46]:
# --- Stage 2: Add context only to top-K numeric bases ---

K = 3  # number of top numeric variants to expand
top_k_numeric = (
    stage1.groupby("Feature_Set", as_index=False)["Mean_F1"]
    .mean()
    .sort_values("Mean_F1", ascending=False)
    .head(K)["Feature_Set"]
    .tolist()
)
print(f"Top {K} numeric bases to expand: {top_k_numeric}")

# Build context-augmented subsets only for those bases
context_blocks = {
    "temporal": ["Month","SpecialDay","Weekend"],
    "user": ["VisitorType"],
    "technical": ["OperatingSystems","Browser","Region","TrafficType"],
}

stage2_sets = {}
for base_name in top_k_numeric:
    base_cols = feature_sets_clean[base_name]
    for ctx_name, ctx_cols in context_blocks.items():
        new_name = f"{base_name}_plus_{ctx_name}"
        stage2_sets[new_name] = base_cols + ctx_cols

# Evaluate those only
rows2 = []
for fs_name, fs_cols in stage2_sets.items():
    pre = make_preprocessor(fs_cols)
    for mdl_name, mdl in models.items():
        pipe = Pipeline([("prep", pre), ("clf", mdl)])
        f1  = cross_val_score(pipe, X_engineered[fs_cols], y, cv=cv, scoring="f1", n_jobs=-1)
        acc = cross_val_score(pipe, X_engineered[fs_cols], y, cv=cv, scoring="accuracy", n_jobs=-1)
        rows2.append({
            "Feature_Set": fs_name, "Model": mdl_name,
            "Mean_F1": f1.mean(), "Std_F1": f1.std(),
            "Mean_Acc": acc.mean(), "Std_Acc": acc.std(),
        })

stage2 = pd.DataFrame(rows2).sort_values(["Model","Mean_F1"], ascending=[True,False])
display(stage2)


Top 3 numeric bases to expand: ['num_pruned_engineered', 'num_behavioral_ratios', 'num_breadth_depth']


,Feature_Set,Model,Mean_F1,Std_F1,Mean_Acc,Std_Acc
0,num_pruned_engineered_plus_temporal,LogReg,0.480373,0.010842,0.882299,0.001076
4,num_pruned_engineered_plus_technical,LogReg,0.473497,0.019079,0.881996,0.002634
2,num_pruned_engineered_plus_user,LogReg,0.469456,0.020097,0.881894,0.002893
10,num_behavioral_ratios_plus_technical,LogReg,0.464208,0.029132,0.884732,0.003516
6,num_behavioral_ratios_plus_temporal,LogReg,0.457546,0.014431,0.883009,0.001999
8,num_behavioral_ratios_plus_user,LogReg,0.451446,0.020842,0.883110,0.002947
16,num_breadth_depth_plus_technical,LogReg,0.051854,0.016654,0.844485,0.002296
12,num_breadth_depth_plus_temporal,LogReg,0.048171,0.019146,0.844485,0.001547
14,num_breadth_depth_plus_user,LogReg,0.033627,0.012723,0.842863,0.001824
1,num_pruned_engineered_plus_temporal,RF,0.650392,0.017046,0.902981,0.002267


In [47]:
# --- Stage 3: Fine-grain test of individual technical features on top of best combo ---

best_base = feature_sets_clean["num_pruned_engineered"] + ["Month","SpecialDay","Weekend"]

individual_techs = ["OperatingSystems","Browser","Region","TrafficType"]
rows3 = []

for tech in individual_techs:
    cols = best_base + [tech]
    pre = make_preprocessor(cols)
    for mdl_name, mdl in models.items():
        pipe = Pipeline([("prep", pre), ("clf", mdl)])
        f1  = cross_val_score(pipe, X_engineered[cols], y, cv=cv, scoring="f1", n_jobs=-1)
        acc = cross_val_score(pipe, X_engineered[cols], y, cv=cv, scoring="accuracy", n_jobs=-1)
        rows3.append({
            "Feature_Set": f"{tech}_only",
            "Model": mdl_name,
            "Mean_F1": f1.mean(), "Std_F1": f1.std(),
            "Mean_Acc": acc.mean(), "Std_Acc": acc.std(),
        })

stage3 = pd.DataFrame(rows3).sort_values(["Model","Mean_F1"], ascending=[True,False])
display(stage3)


,Feature_Set,Model,Mean_F1,Std_F1,Mean_Acc,Std_Acc
6,TrafficType_only,LogReg,0.480409,0.014153,0.881894,0.001659
0,OperatingSystems_only,LogReg,0.478820,0.009968,0.881995,0.001299
2,Browser_only,LogReg,0.478498,0.008594,0.881387,0.001058
4,Region_only,LogReg,0.478109,0.012282,0.881894,0.001308
7,TrafficType_only,RF,0.643280,0.020044,0.902980,0.003160
5,Region_only,RF,0.640062,0.017340,0.901460,0.002118
3,Browser_only,RF,0.639960,0.016997,0.902778,0.002434
1,OperatingSystems_only,RF,0.638665,0.017054,0.900649,0.003817


Here we see they all had lower Mean_F1 compared to our previous model so we do not want to include any of these 

### Pick winners & save plan

In [48]:
# Summaries
stage1_avg = stage1.groupby("Feature_Set", as_index=False).agg(
    Avg_F1=("Mean_F1","mean"), Avg_Acc=("Mean_Acc","mean")
)
best_numeric_name = stage1_avg.sort_values(["Avg_F1","Avg_Acc"], ascending=False).iloc[0]["Feature_Set"]

stage2_avg = stage2.groupby("Feature_Set", as_index=False).agg(
    Avg_F1=("Mean_F1","mean"), Avg_Acc=("Mean_Acc","mean")
)
best_overall_name = stage2_avg.sort_values(["Avg_F1","Avg_Acc"], ascending=False).iloc[0]["Feature_Set"]

# IMPORTANT: stage2_sets holds the *_plus_* combos; feature_sets_clean holds the base sets.
# Merge them so we can look up any set by name.
all_feature_sets = {}
if "feature_sets_clean" in globals():
    all_feature_sets.update(feature_sets_clean)
if "stage2_sets" in globals():
    all_feature_sets.update(stage2_sets)

# Now safe to fetch columns
best_numeric_cols = all_feature_sets[best_numeric_name]
best_overall_cols = all_feature_sets[best_overall_name]

print("==== Selections ====")
print(f"Best numeric engine: {best_numeric_name} | cols: {len(best_numeric_cols)}")
print(f"Best overall combo : {best_overall_name} | cols: {len(best_overall_cols)}")

# Save a plan
plan = {
    "best_numeric": {"name": best_numeric_name, "columns": list(best_numeric_cols)},
    "best_overall": {"name": best_overall_name, "columns": list(best_overall_cols)},
}
os.makedirs(RESULTS_DIR, exist_ok=True)
with open(os.path.join(RESULTS_DIR, "feature_set_plan.json"), "w") as f:
    json.dump(plan, f, indent=2)

stage1_avg.to_csv(os.path.join(RESULTS_DIR, "stage1_numeric_summary.csv"), index=False)
stage2_avg.to_csv(os.path.join(RESULTS_DIR, "stage2_context_summary.csv"), index=False)
print(f"Saved summaries & plan to {RESULTS_DIR}/")


==== Selections ====
Best numeric engine: num_pruned_engineered | cols: 8
Best overall combo : num_pruned_engineered_plus_temporal | cols: 11
Saved summaries & plan to ../results/
